# Elliptical Magnet Array Optimization

Example usage notebook. Adjust parameters for your design.

All measurements in meters (use `inmm = 1e-3` to convert from mm).

## 1. Imports

In [ ]:
import numpy as np
import cupy as cp
from datetime import datetime

# Import the ellipse optimization package
from ellipse_opt import (
    field_calculator,  # GPU field calculation
    sampling,          # DSV sample point generation
    generator,         # Magnet placement and reflection
    optimizer,         # Simulated annealing
    calibration,       # Auto-calibrate SA parameters
    visualization      # Plotting and 3D viewer
)
import visualize_configuration as VS  # Magpylib 3D viewer (optional)
import field_validation as fv         # Field validation (optional)

GRFC = field_calculator
print("Imports OK")

In [ ]:
# ======================================================================
# RANDOM SEED FOR REPRODUCIBILITY
# ======================================================================
RANDOM_SEED = 42  # Set to None for non-reproducible runs

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)
    cp.random.seed(RANDOM_SEED)
    print(f"✓ Random seed set to {RANDOM_SEED} for reproducibility")
else:
    print("⚠ No random seed - results will vary between runs")

## 2. Load Field Data

In [ ]:
# Load precomputed magnetic field lookup table
# Generate your own: precompute/Generate_Precomputed_Field_Batched_analytical.ipynb

PRECOMPUTED_FIELD = 'YOUR_PRECALC_FILE.npz'  # <-- UPDATE THIS

B = np.load(PRECOMPUTED_FIELD)
Precalc_Bxyz_value = B['Bxyzvalue'].astype(np.float32)
Ref_grid = B['Ref_grid']
step_size = float(B['step_size'])
GRFC.load_field(Precalc_Bxyz_value, Ref_grid, step_size)

## 3. Parameters (EXACT from original)

In [ ]:
inmm = 1e-3
halbach_magnet_size = 6.35 * inmm  # 6.35mm cube magnets
layer_clearance = 1.524 * inmm     # Shimstock thickness
sheet_thickness = halbach_magnet_size + layer_clearance

# Geometry parameters (user-defined)
n_magnets_q1 = 2170      # Magnets in half-ellipse (reflected to full)
z_min = 0.0              # Minimum z (half-z symmetry, z >= 0)
z_max = 200 * inmm       # Maximum z (200mm half-length)
rx_inner = 150 * inmm    # Ellipse semi-axis x (150mm)
ry_inner = 125 * inmm    # Ellipse semi-axis y (125mm)
gap = 40 * inmm          # Radial gap for magnets (40mm)
magnet_size = halbach_magnet_size
n_sample_points = 500    # DSV sample points

# DSV (Design Spherical Volume) parameters
dsv_radius = max(rx_inner, ry_inner) * 0.57  # 57% of larger bore dimension
sphere_radius = dsv_radius
z_min_dsv = -(z_max - halbach_magnet_size)
z_max_dsv = z_max - halbach_magnet_size

# Cost function weights
weights = {
    'field_strength': 50000,      # Weight for maximizing field
    'uniformity': 50000*20,       # Weight for field homogeneity (1,000,000)
    'z_barrier': 5.0,             # Soft barrier at z bounds
    'overlap_barrier': 5.0,       # Soft barrier for magnet overlap
    'radius_barrier': 5.0,        # Soft barrier at radial bounds
    'bolt_barrier': 1.0           # Soft barrier for bolt holes
}

def calculate_step_divisors(magnet_size, z_min, z_max, layer_thickness, layer_clearance,
                            r_inner, r_outer, exploration_scale=1.0, phi_resolution_deg=10.0):
    d_min = magnet_size * cp.sqrt(2)
    sheet = layer_thickness + layer_clearance
    r_avg = (r_inner + r_outer) / 2
    z_range = z_max - z_min
    r_range = r_outer - r_inner
    return {
        'z': max(1, int(np.round(z_range / (sheet * exploration_scale)))),
        'r': max(1, int(np.round(r_range / (d_min * exploration_scale)))),
        'theta': max(1, int(np.round((2*np.pi) / ((d_min / r_avg) * exploration_scale)))),
        'phi': max(1, int(np.round(np.pi / (np.radians(phi_resolution_deg) * exploration_scale))))
    }

step_divisors = calculate_step_divisors(
    halbach_magnet_size, z_min, z_max, halbach_magnet_size, layer_clearance,
    (rx_inner + ry_inner) / 2, (rx_inner + ry_inner) / 2 + gap, 4.5, 10.0
)

# SA parameters
cooling_rate = 0.999         # Temperature decay rate
display_interval = 1000      # Print progress every N iterations
initial_temp = 1.0           # Starting temperature (calibration will override)
n_iterations = 200000        # Total iterations
explore_exploit_pct = 0.85   # Switch to transition phase at 85%
fine_tune_pct = 0.92         # Switch to fine-tune phase at 92%

print(f"Step divisors: {step_divisors}")
print(f"Design: {rx_inner*2000:.0f}x{ry_inner*2000:.0f}mm bore, {n_magnets_q1*4} magnets")


## 4. Sample Points

In [ ]:
sample_points = sampling._generate_sample_points_small_dsv(n_sample_points, sphere_radius, z_min_dsv, z_max_dsv)
print(f"Generated {len(sample_points)} DSV sample points")

## 5. Generate Initial Config

In [ ]:
# ======================================================================
# GENERATE INITIAL CONFIGURATION
# ======================================================================
# For a FIRST RUN: Generate random initial configuration
initial_q1 = generator.generate_half_ellipse(n_magnets_q1, z_max, rx_inner, ry_inner, gap, halbach_magnet_size)

# ======================================================================
# FOR A SECOND RUN: Load previous result and continue optimizing
# Uncomment below and comment out the generate_half_ellipse line above:
# ======================================================================
# prev = np.load('ellipse_design_YYYYMMDD_HHMMSS.npz')
# initial_q1 = cp.asarray(prev['q1_config'])
# print(f"Loaded previous result: {len(initial_q1)} magnets")

full_config = generator.reflect_to_half_ellipse(initial_q1)
print(f"Q1: {len(initial_q1)}, Full: {len(full_config)}")

In [ ]:
# Calculate initial field for comparison
B_init, B_mag_init = GRFC.calc_field(full_config, sample_points)
initial_avg_field = float(cp.mean(B_mag_init))
initial_rms_dev = float(cp.std(B_mag_init))  # RMS deviation in mT
initial_homogeneity_pct = initial_rms_dev / initial_avg_field * 100  # Coefficient of variation

print("="*50)
print("INITIAL CONFIGURATION")
print("="*50)
print(f"Field: {initial_avg_field:.2f} mT")
print(f"RMS deviation: {initial_rms_dev:.4f} mT")
print(f"Homogeneity: {initial_homogeneity_pct:.2f}% (CV = std/mean)")
print("="*50)

## 6. Visualize the Intial Configuiration

In [ ]:
# Plot initial half Elliptical configuration
visualization.Plot_positions_plotly(initial_q1, sample_points)

## 7. Calibration 

In [ ]:
# ======================================================================
# RUN CALIBRATION
# ======================================================================

T0, Tf, alpha, diag = calibration.calibrate_sa_rigorous(
    initial_config=initial_q1,
    z_min=z_min,
    z_max=z_max,
    rx_inner=rx_inner,
    ry_inner=ry_inner,
    gap=gap,
    magnet_size=halbach_magnet_size,
    sample_points=sample_points,
    weights=weights,
    step_divisors=step_divisors,
    Precalc_Bxyz_value=Precalc_Bxyz_value,
    Ref_grid=Ref_grid,
    step_size=step_size,
    cooling_rate=cooling_rate,
    n_iterations=100000,  # Temp value for sampling (actual set later)
    n_samples=200,
    chi_0=0.9,
    chi_f=0.01,
    moves_per_level=1,
    hot_temp=1e4,
    cold_ratio=0.01,
)

# Apply calibration
initial_temp = T0
# Apply calibration
cooling_rate = alpha
print(f"\n✓ Applied: initial_temp = {initial_temp:,.0f}, cooling_rate = {cooling_rate:.8f}")

# ======================================================================
# AUTO-CALCULATE ITERATIONS FOR TARGET FINAL TEMPERATURE
# ======================================================================
# Formula: n = log(T_final / T0) / log(cooling_rate)

T_final_target = 1  # Target: near-greedy (acceptance ~1-5%)

# OPTION: Override iteration count (set to None for auto-calculation)
n_iterations_override = 500  # User requested 300k

if n_iterations_override is not None:
    n_iterations = n_iterations_override
    # Recalculate cooling rate to reach T_final in specified iterations
    cooling_rate = (T_final_target / T0) ** (1 / n_iterations)
    print(f"\n>>> OVERRIDE: Using {n_iterations:,} iterations")
    print(f">>> Adjusted cooling_rate = {cooling_rate:.10f}")
    n_iterations_needed = n_iterations
else:
    # Calculate required iterations from calibrated cooling rate
    n_iterations_needed = int(np.log(T_final_target / T0) / np.log(cooling_rate))

print(f"\n{'='*60}")
print(f"AUTO-ITERATION CALCULATION")
print(f"{'='*60}")
print(f"  T0 = {T0:,.0f}")
print(f"  T_final_target = {T_final_target}")
print(f"  cooling_rate = {cooling_rate}")
print(f"  n_iterations needed = {n_iterations_needed:,}")

# Set n_iterations from calibration (unless override is active)
if n_iterations_override is None:
    n_iterations = n_iterations_needed
    print(f"  → Using auto-calculated n_iterations = {n_iterations:,}")
else:
    print(f"  → Using OVERRIDE n_iterations = {n_iterations:,}")

# Verify final temperature
T_final_actual = T0 * (cooling_rate ** n_iterations)
print(f"  T_final_actual = {T_final_actual:.2f}")
print(f"{'='*60}")


## 8. Run Optimization

In [ ]:
best_q1, best_full, best_cost, history = optimizer.simulated_annealing_half_ellipse(
    initial_q1, z_min, z_max, rx_inner, ry_inner, gap,
    halbach_magnet_size, sample_points, weights,
    Precalc_Bxyz_value, Ref_grid, step_size, step_divisors,
    initial_temp, cooling_rate, n_iterations, display_interval,
    explore_exploit_pct=explore_exploit_pct, fine_tune_pct=fine_tune_pct
)
print("Done!")

## 9. Results

In [ ]:
visualization.plot_cost_progression(history)

B_field, B_mag = GRFC.calc_field(best_full, sample_points)
avg_field = float(cp.mean(B_mag))
rms_dev = float(cp.std(B_mag))  # RMS deviation in mT
homogeneity_pct = rms_dev / avg_field * 100  # Coefficient of variation

print(f"Field: {avg_field:.2f} mT")
print(f"RMS deviation: {rms_dev:.4f} mT")

## 10. Save

In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
np.savez(f"ellipse_design_{ts}.npz",
    q1_config=cp.asnumpy(best_q1), full_config=cp.asnumpy(best_full),
    cost=float(best_cost), field_mT=avg_field, rms_dev_mT=rms_dev
)
print(f"Saved!")
print(f"Field: {avg_field:.2f} mT")
print(f"RMS deviation: {rms_dev:.4f} mT")

## 11. Visualization & Comparison

In [ ]:
# 3D Visualization - displays automatically in notebook
# If WebGL fails, use: visualization.show_magnets_plotly(config, size, show=True)
# This will automatically open in browser as fallback

# Option 1: Built-in Plotly viewer (no magpylib required)
# 
visualization.show_magnets_plotly(cp.asnumpy(best_full), halbach_magnet_size)

# Option 2: Magpylib viewer (more mature, requires magpylib)
# import visualize_configuration as VS
# view = VS.rad_halbach_Design_rad(halbach_magnet_size, cp.asnumpy(best_full))
# view.show()

In [ ]:
# Comprehensive Configuration Visualization (requires visualize_configuration module)
VS.visualize_configuration(
    config=cp.asnumpy(best_full),
    n_magnets=len(best_full),
    z_min=-z_max,
    z_max=z_max,
    r_min=rx_inner,
    vmax_field=80,
    magnet_size=halbach_magnet_size,
    sample_points=cp.asnumpy(sample_points),
    sphere_radius=sphere_radius,
    Precalc_Bxyz_value=cp.asnumpy(Precalc_Bxyz_value),
    Ref_grid=Ref_grid,
    step_size=step_size
)

In [ ]:
# Field Validation Module - Magpylib-based validation
# Uses independent analytical calculation to cross-validate GPU precomputed field
import field_validation as fv
import magpylib as magpy
import matplotlib.pyplot as plt

In [ ]:
# Overlay: Initial vs Optimized Field Magnitude (Magpylib Validation)
# Define axis range (adjust as needed)
axis_range = sphere_radius  # Use DSV radius, or set custom e.g. 0.07

# Create magpylib views
initial_full = generator.reflect_to_half_ellipse(initial_q1)
initial_view = VS.rad_halbach_Design_rad(halbach_magnet_size, cp.asnumpy(initial_full))
optimized_view = VS.rad_halbach_Design_rad(halbach_magnet_size, cp.asnumpy(best_full))

# Use module function for overlay comparison
fv.overlay_comparison(initial_view, optimized_view, axis_range=axis_range,
                      initial_label="Initial Random Input Design",
                      optimized_label="Optimized Design");